<a href="https://colab.research.google.com/github/mayajdias/walmart-hurricane-project/blob/main/notebooks/Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning

## Decision Log

In [3]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

## Loading Data / EDA: distribution_centers.csv

In [4]:
import pandas as pd, numpy as np
distribution = pd.read_csv('distribution_centers.csv')
distribution.head()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


In [5]:
distribution.shape

(5, 8)

5 observations, 8 columns

In [6]:
distribution.dtypes

,0
dc_id,object
dc_name,object
city,object
state,object
avg_daily_shipments,int64
surge_capacity,object
distance_to_orlando_mi,int64
cost_per_shipment,float64


Need to change surge_capacity to int

In [7]:
distribution.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dc_id                   5 non-null      object 
 1   dc_name                 5 non-null      object 
 2   city                    5 non-null      object 
 3   state                   5 non-null      object 
 4   avg_daily_shipments     5 non-null      int64  
 5   surge_capacity          5 non-null      object 
 6   distance_to_orlando_mi  5 non-null      int64  
 7   cost_per_shipment       4 non-null      float64
dtypes: float64(1), int64(2), object(5)
memory usage: 452.0+ bytes


In [8]:
distribution.describe()

,avg_daily_shipments,distance_to_orlando_mi,cost_per_shipment
count,5.00000,5.000000,4.000000
mean,360.00000,485.000000,410.625000
std,96.17692,377.987434,197.064023
min,250.00000,141.000000,245.000000
25%,300.00000,235.000000,284.750000
50%,350.00000,438.000000,355.250000
75%,400.00000,503.000000,481.125000
max,500.00000,1108.000000,687.000000


In [9]:
distribution.isnull().sum()

,0
dc_id,0
dc_name,0
city,0
state,0
avg_daily_shipments,0
surge_capacity,0
distance_to_orlando_mi,0
cost_per_shipment,1


1 null item in cost_per_shipment

In [10]:
distribution["state"].value_counts()

,count
state,
FL,2
GA,1
TX,1
AL,1


Cleaning Required: remove non-numerical values from surge_capacity and make integer type. Address missing value in cost_per_shipment.

## Cleaning: distribution_centers.csv

In [11]:
def clean_distributions(df):
  df = df.copy()
  if "surge_capacity" in df.columns:
    df["surge_capacity"] = df["surge_capacity"].str.extract(r"(\d+)").astype(int)
  if "cost_per_shipment" in df.columns:
        mean_cost_per_shipment = df["cost_per_shipment"].mean()
        df["cost_per_shipment"] = df["cost_per_shipment"].fillna(mean_cost_per_shipment)
  return df

In [12]:
distributions_cleaned = clean_distributions(distribution)
distributions_cleaned.head()

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.000
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.500
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.000
3,DC-MIA,Miami Distribution Center,Miami,FL,350,750,235,298.000
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,410.625


In [13]:
log("Cleaning", "converted surge_capacity to integer and addressed missing value in cost_per_shipment", "5")

[Cleaning] converted surge_capacity to integer and addressed missing value in cost_per_shipment (5 row(s))


## Loading Data / EDA: store_locations.csv

In [14]:
stores = pd.read_csv('store_locations.csv')
stores.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


In [15]:
stores.shape

(14, 8)

14 observations, 8 columns

In [16]:
stores.dtypes

,0
store_id,object
store_name,object
city,object
state,object
zip_code,int64
latitude,float64
longitude,float64
region,object


In [17]:
stores.duplicated().sum()

np.int64(0)

In [18]:
stores.duplicated(subset=["store_id"]).sum()

np.int64(0)

In [19]:
stores.duplicated(subset=["store_name"]).sum()

np.int64(2)

There are two duplicate store names

In [20]:
name_duplicates = stores[stores.duplicated(subset=["store_name"], keep=False)]
print(name_duplicates)

   store_id                             store_name       city    state  \
0    FL-239            Walmart Supercenter Orlando    Orlando       FL   
4    FL-287  Walmart Neighborhood Market Kissimmee  Kissimmee  Florida   
12    FL239            Walmart Supercenter Orlando    Orlando       FL   
13   FL_287  Walmart Neighborhood Market Kissimmee  Kissimmee  Florida   

    zip_code  latitude  longitude   region  
0      32801   28.5383   -81.3792  Central  
4      34741   28.2920   -81.4076  Central  
12     32801   28.5383   -81.3792  Central  
13     34741   28.2920   -81.4076  Central  


The two duplicated store names are not listed as total duplicates because store_id and state are formatted differently between them. Need to standardize format for these columns, then drop duplicate rows.

In [21]:
stores["city"].value_counts()

,count
city,
Orlando,2
Kissimmee,2
Miami,1
Tampa,1
Jacksonville,1
Daytona Beach,1
Fort Myers,1
Gainesville,1
Tallahassee,1


In [22]:
stores["state"].value_counts()

,count
state,
FL,10
Florida,3
Fla.,1


Need to standardize state name - all are in Florida, but formatted 3 ways. Should use "FL" because this is the convention in the previous data set.

In [23]:
stores["region"].value_counts()

,count
region,
Central,5
West,3
North,2
South,1
East,1
Panhandle,1
Panhandle,1


In [24]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    14 non-null     object 
 1   store_name  14 non-null     object 
 2   city        14 non-null     object 
 3   state       14 non-null     object 
 4   zip_code    14 non-null     int64  
 5   latitude    13 non-null     float64
 6   longitude   13 non-null     float64
 7   region      14 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 1.0+ KB


In [25]:
stores.describe()

,zip_code,latitude,longitude
count,14.000000,13.000000,13.000000
mean,33200.428571,28.418192,-82.069631
std,893.995759,1.319349,1.676108
min,32099.000000,25.761700,-87.216900
25%,32526.000000,27.950600,-82.324800
50%,32951.000000,28.292000,-81.655700
75%,33776.000000,29.210800,-81.379200
max,34741.000000,30.421300,-80.191800


In [26]:
stores.isnull().sum()

,0
store_id,0
store_name,0
city,0
state,0
zip_code,0
latitude,1
longitude,1
region,0


In [27]:
stores[stores['latitude'].isna()]

,store_id,store_name,city,state,zip_code,latitude,longitude,region
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle


In [28]:
stores[stores['longitude'].isna()]

,store_id,store_name,city,state,zip_code,latitude,longitude,region
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle


The missing values for latitude and longitude are both in the same row, the Walmart Supercenter Tallahassee.

Cleaning Required: standardize store_id format by removing filler characters ("-", "_", etc.). Standardize state name by setting all to "FL". Address missing latitude and longitude for Tallahassee store. Drop duplicate rows.

## Cleaning: store_locations.csv

In [29]:
def clean_stores(df):
  df = df.copy()
  if 'store_id' in df.columns:
        df['store_id'] = df['store_id'].astype(str).str.replace(r'[^a-zA-Z0-9]', '', regex=True)
  if 'state' in df.columns:
        df['state'] = 'FL'
  if 'latitude' in df.columns:
        tallahassee_latitude = 30.4383
        df['latitude'] = df['latitude'].fillna(tallahassee_latitude)
  if 'longitude' in df.columns:
        tallahassee_longitude = -84.2807
        df['longitude'] = df['longitude'].fillna(tallahassee_longitude)
  df = df.drop_duplicates()
  return df

In [30]:
stores_cleaned = clean_stores(stores)
stores_cleaned.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL287,Walmart Neighborhood Market Kissimmee,Kissimmee,FL,34741,28.2920,-81.4076,Central


In [31]:
log("Cleaning", "standardized store_id and state, filled missing latitude and longitude", "14")

[Cleaning] standardized store_id and state, filled missing latitude and longitude (14 row(s))


In [32]:
stores_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    12 non-null     object 
 1   store_name  12 non-null     object 
 2   city        12 non-null     object 
 3   state       12 non-null     object 
 4   zip_code    12 non-null     int64  
 5   latitude    12 non-null     float64
 6   longitude   12 non-null     float64
 7   region      12 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 864.0+ bytes


## Loading Data / EDA: walmart_transactions.csv

In [33]:
transactions = pd.read_csv("walmart_transactions.csv")
transactions.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


Need to standardize store_id

In [34]:
transactions.shape

(25562, 8)

25,562 observations, 8 columns

In [35]:
transactions.duplicated().sum()

np.int64(30)

30 duplicate rows even before cleaning

In [36]:
transactions.dtypes

,0
transaction_id,object
store_id,object
timestamp,object
sku,object
product_name,object
category,object
quantity,int64
unit_price,object


Need to make unit_price a float and timestamp a date

In [37]:
transactions["sku"].value_counts()

,count
sku,
TARP-HD,766
LED-FLASH,732
FLASH-01,728
HDTARP-01,701
CHRG-PORT,684
CANDLE-EMRG,664
CNDL-EM,652
BEER-24,643
FIRSTAID-KIT,637


Need to consolidate SKUs

### Ambiguous SKU investigation:

In [42]:
contains_STREAM = transactions[transactions['sku'].str.lower().str.contains('stream', na=False)]
contains_STREAM.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
34,TXN019961,FL-445,09-02-2024 07:05:04,STREAM-01,Roku Streaming Stick,Electronics,1,30.70
37,TXN021031,FL-501,2024-09-04T10:07:24,STREAM-01,STREAMING STICK,Electronics,1,28.65
81,TXN004193,FL-330,2024-09-18 12:03:52,STREAM-01,Streaming Device,electronics,1,30.94
162,TXN022256,FL-078,2024-09-06T20:26:50,STREAM-01,Streaming Device,Electronics,1,29.18
308,TXN009170,FL-710,2024-09-22T11:40:05,STREAM-01,STREAMING STICK,ELECTRONICS,1,28.70


SKU STREAM_01 is a Roku Streaming Stick

In [43]:
contains_TP = transactions[transactions['sku'].str.lower().str.contains('tp-12', na=False)]
contains_TP.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
45,TXN021737,FL-619,NaN,TP-12,TOILET PAPER 12PK,HOUSEHOLD,2,9.74
48,TXN013700,FL-156,09/25/2024 17:38,TP-12,Toilet Paper 12-Roll,Household,2,9.93
82,TXN020295,fl-412,2024-09-03 20:13:58,TP-12,TOILET PAPER 12PK,Household,1,9.60
103,TXN012143,FL-619,2024-09-24 17:41:44,TP-12,TP 12 Pack,Household Essentials,1,9.68
323,TXN016091,FL-105,2024-09-27 09:49:38,TP-12,Toilet Paper 12-Roll,Household Essentials,2,10.16


SKU TP-12 is toilet paper

In [44]:
contains_FAID = transactions[transactions['sku'].str.lower().str.contains('faid-01', na=False)]
contains_FAID.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
110,TXN011942,FL-078,2024-09-24 17:11:45,FAID-01,FIRST AID KIT,HEALTH,2,14.41
170,TXN005774,FL-105,09/20/2024 21:12,FAID-01,First Aid Kit,health,1,14.97
173,TXN017029,FL-330,09/28/2024 06:50,FAID-01,FIRST AID KIT,health,2,15.17
200,TXN013846,FL-501,09/25/2024 10:18,FAID-01,First Aid Kit,HEALTH,2,14.77
204,TXN022216,FL-501,2024-09-06T15:21:16,FAID-01,First-Aid Kit,Health,1,$15.05


SKU FAID-01 is a first aid kit

In [45]:
contains_PB = transactions[transactions['sku'].str.lower().str.contains('pb-16', na=False)]
contains_PB.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
26,TXN018977,FL-105,09/01/2024 06:26,PB-16,Peanut Butter 16oz,Groceries,1,3.22
102,TXN004056,FL-239,2024-09-18 08:01:34,PB-16,Peanut Butter,Groceries,1,3.23
238,TXN023524,FL-619,08-Sep-2024 16:13,PB-16,Peanut Butter,Groceries,1,$3.38
367,TXN016207,FL-412,09/27/2024 10:16,PB-16,Peanut Butter 16oz,grocery,2,3.32
395,TXN016071,FL105,2024-09-27 15:05:48,PB-16,Peanut Butter,Groceries,1,3.32


SKU PB-16 is peanut butter

In [46]:
contains_PT = transactions[transactions['sku'].str.lower().str.contains('pt-12', na=False)]
contains_PT.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
47,TXN005801,FL-330,2024-09-20 21:23:43,PT-12,Pop Tarts - Strawberry,BREAKFAST,3,3.49
138,TXN012731,FL_239,NaN,PT-12,POPTART-STRAWBERRY,BREAKFAST,4,3.56
227,TXN023448,FL-078,09/08/2024 06:10,PT-12,Strawberry Pop-Tart,Breakfast Foods,3,3.53
249,TXN018956,FL-105,2024-09-01 15:21:52,PT-12,Strawberry Pop-Tart,Breakfast Foods,4,$3.39
264,TXN019047,FL-412,01-Sep-2024 08:02,PT-12,Pop-Tarts Strawberry,BREAKFAST,3,3.57


SKU PT-12 is strawberry poptarts

In [47]:
contains_459812 = transactions[transactions['sku'].str.lower().str.contains('459812', na=False)]
contains_459812.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
27,TXN007697,FL-710,21-Sep-2024 19:46,SKU#459812,POPTART-STRAWBERRY,BREAKFAST,4,3.45
142,TXN003351,FL-330,2024-09-17 19:25:29,SKU#459812,Pop-Tarts Strawberry,Breakfast,4,3.66
180,TXN009599,FL-412,2024-09-23 15:39:21,SKU#459812,Pop-Tarts Strawberry,Breakfast,3,3.51
197,TXN012728,FL-239,2024-09-25 12:34:01,SKU#459812,pop-tarts strawberry,breakfast,1,3.58
202,TXN010186,FL-619,23-Sep-2024 17:44,SKU#459812,Strawberry Pop-Tart,breakfast,3,3.33


SKU SKU#459812 is strawberry poptarts

In [38]:
transactions.isnull().sum()

,0
transaction_id,0
store_id,0
timestamp,886
sku,0
product_name,0
category,0
quantity,0
unit_price,0


timestamp is the only column with missing values, it has 886 missing values

In [40]:
transactions["category"].value_counts()

,count
category,
EMERGENCY,1823
Emergency Supplies,1819
emergency,1789
Emergency,1764
Groceries,859
electronics,857
Electronics,845
Grocery,837
ELECTRONICS,824


Need to consolidate category names

## Cleaning: walmart_transactions.csv

### Mapping SKUs

In [48]:
canonical_sku_groups = {
    'POPTART-STRAW': ['PT-12', 'SKU#459812', 'PT_STRAWBERRY'],
    'SOUP-CAN': ['CNSOUP-01'],
    'BG-MONO': ['GAME-MONO', 'MONOPOLY-01'],
    'BEER-24': ['LTBEER24'],
    'WTR-24': ['WTR_24PK', 'WATER24PK'],
    'CANDLE-EMRG': ['CNDL-EM'],
    'TARP-HD': ['HDTARP-01'],
    'LED-FLASH': ['FLASH-01'],
    'CHRG-PORT': ['PORTCHARGER'],
    'FIRSTAID-KIT': ['FAID-01'],
    'ROKU-STD': ['STREAM-01'],
    'PTOWEL-6': ['PAPERTOWEL6'],
    'PLAYCARD-STD': ['CARDS-01'],
    'BREAD-WH': ['WHITEBREAD'],
    'TOILETPAPER12': ['TP-12'],
    'PNUTBTR-16': ['PB-16'],
    'BABYFORMULA': ['BFORM-01'],
    'DIAPERS-3': ['DIAP-S3'],
    'BAT-AA4': ['BATT-100', 'BATT_AA_4PK'],
    'CHIPVARIETY': ['CHIPS-VP']
}

sku_mapping = {
    original: target
    for target, originals in canonical_sku_groups.items()
    for original in originals
}

### Mapping categories

In [50]:
canonical_category_groups = {
    'Emergency': ['EMERGENCY', 'Emergency Supplies', 'emergency'],
    'Grocery': ['Groceries', 'grocery', 'GROCERY'],
    'Electronics': ['electronics', 'ELECTRONICS'],
    'Entertainment': ['Games & Entertainment', 'entertainment', 'ENTERTAINMENT'],
    'Household': ['HOUSEHOLD', 'household', 'Household Essentials'],
    'Baby': ['baby', 'BABY', 'Baby Products'],
    'Breakfast': ['breakfast', 'BREAKFAST', 'Breakfast Foods'],
    'Alcohol': ['ALCOHOL', 'Alcoholic Beverages'],
    'Snacks': ['snacks', 'SNACKS'],
    'Health': ['HEALTH', 'health', 'Health & Wellness']
}

category_mapping = {
    original: target
    for target, originals in canonical_category_groups.items()
    for original in originals
}

### Applying SKU mapping

In [51]:
def consolidate_skus(df):
    df = df.copy()
    if 'sku' in df.columns:
        df['canonical_sku'] = df['sku'].map(sku_mapping).fillna(df['sku'])
    return df

transactions_skus_consolidated = consolidate_skus(transactions)
transactions_skus_consolidated.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,PNUTBTR-16
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,SOUP-CAN
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,TOILETPAPER12
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,ROKU-STD
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,BEER-24


In [52]:
log("Cleaning", "created canonical_sku column consolidating SKUs", "25562")

[Cleaning] created canonical_sku column consolidating SKUs (25562 row(s))


### Applying category mapping

In [53]:
def consolidate_categories(df):
    df = df.copy()
    if 'category' in df.columns:
        df['canonical_category'] = df['category'].map(category_mapping).fillna(df['category'])
    return df

transactions_fully_consolidated = consolidate_categories(transactions_skus_consolidated)
transactions_fully_consolidated.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku,canonical_category
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,PNUTBTR-16,Grocery
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,SOUP-CAN,Grocery
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,TOILETPAPER12,Household
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,ROKU-STD,Electronics
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,BEER-24,Alcohol


In [54]:
log("Cleaning", "created canonical_category column consolidating categories", "25562")

[Cleaning] created canonical_category column consolidating categories (25562 row(s))


### Cleaning unit_price and store_id columns

In [55]:
def clean_transactions(df):
  df = df.copy()
  if 'store_id' in df.columns:
        df['store_id'] = (df['store_id'].astype(str).str.upper().str.replace(r'[^A-Z0-9]', '', regex=True))
  if 'unit_price' in df.columns:
        df['unit_price'] = (df['unit_price'].astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float))
  df = df.drop_duplicates().reset_index(drop=True)
  return df

In [56]:
print("Length before:", len(transactions_fully_consolidated))
transactions_cleaned_price_id = clean_transactions(transactions_fully_consolidated)
print("Length after:", len(transactions_cleaned_price_id))
print("Rows removed:", len(transactions_fully_consolidated) - len(transactions_cleaned_price_id))
transactions_cleaned_price_id.head()

Length before: 25562
Length after: 25532
Rows removed: 30


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku,canonical_category
0,TXN012636,FL710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,PNUTBTR-16,Grocery
1,TXN000759,FL412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,SOUP-CAN,Grocery
2,TXN003338,FL105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,TOILETPAPER12,Household
3,TXN025020,FL330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,ROKU-STD,Electronics
4,TXN015797,FL302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,BEER-24,Alcohol


In [57]:
log("Cleaning", "cleaned and standardized store_id and unit_price", "25562")

[Cleaning] cleaned and standardized store_id and unit_price (25562 row(s))


In [58]:
log("Cleaning", "dropped duplicate rows", "30")

[Cleaning] dropped duplicate rows (30 row(s))


### Cleaning timestamp column

In [59]:
def clean_timestamp(df):
    df = df.copy()
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', errors='coerce')
        df = df.dropna(subset=['timestamp'])
    return df

In [60]:
print("Length before:", len(transactions_cleaned_price_id))
transactions_fully_cleaned = clean_timestamp(transactions_cleaned_price_id)
print("Length after:", len(transactions_fully_cleaned))
print("Rows removed:", len(transactions_cleaned_price_id) - len(transactions_fully_cleaned))
transactions_fully_cleaned.head()

Length before: 25532
Length after: 24648
Rows removed: 884


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku,canonical_category
0,TXN012636,FL710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15,PNUTBTR-16,Grocery
1,TXN000759,FL412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02,SOUP-CAN,Grocery
2,TXN003338,FL105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47,TOILETPAPER12,Household
3,TXN025020,FL330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59,ROKU-STD,Electronics
4,TXN015797,FL302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82,BEER-24,Alcohol


In [61]:
log("Cleaning", "converted timestamp column to consistent datetime format", "25532")

[Cleaning] converted timestamp column to consistent datetime format (25532 row(s))


In [62]:
log("Cleaning", "removed rows with missing timestamp", "884")

[Cleaning] removed rows with missing timestamp (884 row(s))
